In [ ]:
# Rolling Out-of-Sample Portfolio Optimization

This notebook implements the core portfolio optimization and out-of-sample backtesting framework.

Four strategies are compared:

- Equal Weight
- Global Minimum Variance (GMV)
- Maximum Sharpe Ratio
- Risk Parity

The baseline design uses:

- 252 trading days for parameter estimation
- 21 trading days for the holding period
- Long-only, fully invested portfolios
- 10 bps transaction cost per unit of turnover
- Natural portfolio-weight drift between rebalancing dates

Only information available before each rebalance date is used to estimate portfolio weights, preventing look-ahead bias.

The notebook also evaluates robustness to transaction costs and alternative estimation-window lengths.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.optimize import minimize

# ==========================================
# 1. Load prices
# ==========================================

data_dir = Path("../data")
output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

prices = pd.read_csv(
    data_dir / "adjusted_close.csv",
    index_col=0,
    parse_dates=True
)

returns = prices.pct_change().dropna()

assets = prices.columns.tolist()
N = len(assets)

ESTIMATION_WINDOW = 252
HOLDING_PERIOD = 21
TRANSACTION_COST = 0.001

print("Assets:", assets)
print("Return observations:", len(returns))


# ==========================================
# 2. Portfolio optimizers
# ==========================================

def get_weights(method, window_returns):

    n = window_returns.shape[1]

    mu = (
        window_returns.mean().values
        * 252
    )

    cov = (
        window_returns.cov().values
        * 252
    )

    # Small numerical stabilization
    cov = cov + np.eye(n) * 1e-8

    x0 = np.repeat(1 / n, n)

    bounds = [
        (0.0, 1.0)
        for _ in range(n)
    ]

    constraints = {
        "type": "eq",
        "fun": lambda w: np.sum(w) - 1
    }

    # --------------------------
    # Equal Weight
    # --------------------------

    if method == "EqualWeight":

        return x0


    # --------------------------
    # Global Minimum Variance
    # --------------------------

    elif method == "GMV":

        def objective(w):
            return w @ cov @ w

        result = minimize(
            objective,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000}
        )


    # --------------------------
    # Maximum Sharpe
    # rf = 0 at this stage
    # --------------------------

    elif method == "MaxSharpe":

        def objective(w):

            port_return = w @ mu

            port_vol = np.sqrt(
                w @ cov @ w
            )

            if port_vol <= 0:
                return 1e6

            return -port_return / port_vol

        result = minimize(
            objective,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000}
        )


    # --------------------------
    # Risk Parity
    # Equal risk contribution
    # --------------------------

    elif method == "RiskParity":

        def objective(w):

            portfolio_variance = (
                w @ cov @ w
            )

            if portfolio_variance <= 0:
                return 1e6

            marginal_variance = (
                cov @ w
            )

            risk_contribution = (
                w
                * marginal_variance
                / portfolio_variance
            )

            target = np.repeat(
                1 / n,
                n
            )

            return np.sum(
                (
                    risk_contribution
                    - target
                ) ** 2
            )

        result = minimize(
            objective,
            x0,
            method="SLSQP",
            bounds=bounds,
            constraints=constraints,
            options={"maxiter": 1000}
        )

    else:

        raise ValueError(
            f"Unknown method: {method}"
        )


    # Safe fallback
    if (
        not result.success
        or np.any(np.isnan(result.x))
    ):
        return x0

    w = np.maximum(
        result.x,
        0
    )

    return w / w.sum()


# ==========================================
# 3. Rolling out-of-sample backtest
# ==========================================

def rolling_backtest(
    returns,
    method,
    estimation_window=252,
    holding_period=21,
    transaction_cost=0.001
):

    wealth = 1.0

    current_weights = None

    records = []
    weight_records = []

    for start in range(
        estimation_window,
        len(returns),
        holding_period
    ):

        # IMPORTANT:
        # Only past data are used here.
        estimation_data = returns.iloc[
            start - estimation_window:start
        ]

        target_weights = get_weights(
            method,
            estimation_data
        )

        rebalance_date = (
            returns.index[start]
        )

        # --------------------------
        # Turnover
        # --------------------------

        if current_weights is None:

            turnover = 0.0

        else:

            turnover = (
                0.5
                * np.abs(
                    target_weights
                    - current_weights
                ).sum()
            )

        cost_fraction = (
            transaction_cost
            * turnover
        )

        current_weights = (
            target_weights.copy()
        )

        weight_records.append(
            pd.Series(
                target_weights,
                index=assets,
                name=rebalance_date
            )
        )

        # --------------------------
        # Hold for next 21 days
        # --------------------------

        end = min(
            start + holding_period,
            len(returns)
        )

        for i in range(start, end):

            daily_asset_returns = (
                returns.iloc[i].values
            )

            gross_portfolio_return = float(
                current_weights
                @ daily_asset_returns
            )

            # Transaction cost only
            # on first day after rebalance
            if i == start:

                net_portfolio_return = (
                    (1 - cost_fraction)
                    * (
                        1
                        + gross_portfolio_return
                    )
                    - 1
                )

            else:

                net_portfolio_return = (
                    gross_portfolio_return
                )

            wealth *= (
                1
                + net_portfolio_return
            )

            records.append(
                {
                    "Date":
                        returns.index[i],

                    "Wealth":
                        wealth,

                    "Return":
                        net_portfolio_return,

                    "Turnover":
                        turnover
                        if i == start
                        else 0.0
                }
            )

            # Allow weights to drift
            current_weights = (
                current_weights
                * (
                    1
                    + daily_asset_returns
                )
            )

            current_weights = (
                current_weights
                / current_weights.sum()
            )

    backtest = (
        pd.DataFrame(records)
        .set_index("Date")
    )

    weights = pd.DataFrame(
        weight_records
    )

    return backtest, weights


# ==========================================
# 4. Performance metrics
# ==========================================

def portfolio_metrics(backtest):

    wealth = backtest["Wealth"]

    r = (
        backtest["Return"]
        .dropna()
    )

    years = (
        wealth.index[-1]
        - wealth.index[0]
    ).days / 365.25

    cagr = (
        wealth.iloc[-1]
        / 1.0
    ) ** (1 / years) - 1

    annual_vol = (
        r.std()
        * np.sqrt(252)
    )

    sharpe_rf0 = (
        r.mean()
        / r.std()
        * np.sqrt(252)
    )

    downside_returns = np.minimum(
        r,
        0
    )

    downside_deviation = (
        np.sqrt(
            np.mean(
                downside_returns ** 2
            )
        )
        * np.sqrt(252)
    )

    sortino_rf0 = (
        r.mean()
        * 252
        / downside_deviation
    )

    drawdown = (
        wealth
        / wealth.cummax()
        - 1
    )

    max_drawdown = (
        drawdown.min()
    )

    q05 = r.quantile(0.05)

    var95 = -q05

    cvar95 = -r[
        r <= q05
    ].mean()

    annual_turnover = (
        backtest["Turnover"].sum()
        / years
    )

    return pd.Series(
        {
            "CAGR":
                cagr,

            "Annualized_Volatility":
                annual_vol,

            "Sharpe_rf0":
                sharpe_rf0,

            "Sortino_rf0":
                sortino_rf0,

            "Maximum_Drawdown":
                max_drawdown,

            "Daily_VaR_95":
                var95,

            "Daily_CVaR_95":
                cvar95,

            "Annualized_Turnover":
                annual_turnover
        }
    )


# ==========================================
# 5. Run all strategies
# ==========================================

strategies = [
    "EqualWeight",
    "GMV",
    "MaxSharpe",
    "RiskParity"
]

backtests = {}
weights_history = {}

for strategy in strategies:

    print(
        f"Running {strategy}..."
    )

    bt, w = rolling_backtest(
        returns,
        method=strategy,
        estimation_window=ESTIMATION_WINDOW,
        holding_period=HOLDING_PERIOD,
        transaction_cost=TRANSACTION_COST
    )

    backtests[strategy] = bt
    weights_history[strategy] = w


# ==========================================
# 6. Compare strategies
# ==========================================

metrics = pd.DataFrame(
    {
        strategy:
            portfolio_metrics(
                backtests[strategy]
            )

        for strategy
        in strategies
    }
).T

print(
    "\n=== Out-of-Sample Performance ==="
)

display(
    metrics.round(4)
)


# ==========================================
# 7. Average allocation
# ==========================================

average_weights = pd.DataFrame(
    {
        strategy:
            weights_history[
                strategy
            ].mean()

        for strategy
        in strategies
    }
).T

print(
    "\n=== Average Portfolio Weights ==="
)

display(
    average_weights.round(3)
)


# ==========================================
# 8. Concentration diagnostics
# ==========================================

concentration = {}

for strategy in strategies:

    w = weights_history[
        strategy
    ]

    concentration[
        strategy
    ] = pd.Series(
        {
            "Average_Max_Weight":
                w.max(axis=1).mean(),

            "Median_Max_Weight":
                w.max(axis=1).median(),

            "Max_Observed_Weight":
                w.max(axis=1).max()
        }
    )

concentration = pd.DataFrame(
    concentration
).T

print(
    "\n=== Portfolio Concentration ==="
)

display(
    concentration.round(3)
)


# ==========================================
# 9. Save results
# ==========================================

metrics.to_csv(
    output_dir
    / "oos_strategy_metrics.csv"
)

average_weights.to_csv(
    output_dir
    / "average_strategy_weights.csv"
)

concentration.to_csv(
    output_dir
    / "strategy_concentration.csv"
)

for strategy in strategies:

    backtests[
        strategy
    ].to_csv(
        output_dir
        / f"{strategy}_oos_backtest.csv"
    )

    weights_history[
        strategy
    ].to_csv(
        output_dir
        / f"{strategy}_weights.csv"
    )


print(
    "\nOOS period:"
)

print(
    backtests["EqualWeight"].index.min(),
    "→",
    backtests["EqualWeight"].index.max()
)

print(
    "\nNumber of rebalance dates:",
    len(
        weights_history[
            "EqualWeight"
        ]
    )
)

Assets: ['SPY', 'QQQ', 'IWM', 'EFA', 'EEM', 'TLT', 'GLD', 'VNQ']
Return observations: 4023
Running EqualWeight...
Running GMV...
Running MaxSharpe...
Running RiskParity...

=== Out-of-Sample Performance ===


,CAGR,Annualized_Volatility,Sharpe_rf0,Sortino_rf0,Maximum_Drawdown,Daily_VaR_95,Daily_CVaR_95,Annualized_Turnover
EqualWeight,0.0915,0.1323,0.7290,1.0213,-0.2719,0.0123,0.0195,0.1528
GMV,0.0840,0.0889,0.9531,1.3595,-0.2578,0.0088,0.0128,0.6314
MaxSharpe,0.0921,0.1306,0.7415,1.0428,-0.2910,0.0127,0.0201,2.8872
RiskParity,0.0901,0.1210,0.7748,1.0887,-0.2657,0.0113,0.0180,0.4502



=== Average Portfolio Weights ===


,SPY,QQQ,IWM,EFA,EEM,TLT,GLD,VNQ
EqualWeight,0.125,0.125,0.125,0.125,0.125,0.125,0.125,0.125
GMV,0.304,0.006,0.018,0.065,0.011,0.386,0.201,0.009
MaxSharpe,0.179,0.240,0.057,0.038,0.006,0.211,0.203,0.066
RiskParity,0.116,0.100,0.092,0.108,0.091,0.160,0.226,0.107



=== Portfolio Concentration ===


,Average_Max_Weight,Median_Max_Weight,Max_Observed_Weight
EqualWeight,0.125,0.125,0.125
GMV,0.466,0.470,0.605
MaxSharpe,0.591,0.535,1.000
RiskParity,0.275,0.265,0.415



OOS period:
2011-01-04 00:00:00 → 2025-12-31 00:00:00

Number of rebalance dates: 180


In [2]:
# ==========================================
# 10. Transaction-cost robustness
# ==========================================

cost_levels = {
    "0bps": 0.0000,
    "10bps": 0.0010,
    "25bps": 0.0025,
    "50bps": 0.0050
}

cost_results = []

for cost_name, cost in cost_levels.items():

    print(f"Running cost scenario: {cost_name}")

    for strategy in strategies:

        bt, _ = rolling_backtest(
            returns,
            method=strategy,
            estimation_window=252,
            holding_period=21,
            transaction_cost=cost
        )

        m = portfolio_metrics(bt)

        cost_results.append({
            "Transaction_Cost": cost_name,
            "Strategy": strategy,
            "CAGR": m["CAGR"],
            "Sharpe_rf0": m["Sharpe_rf0"],
            "Maximum_Drawdown": m["Maximum_Drawdown"],
            "Annualized_Turnover": m["Annualized_Turnover"]
        })

cost_robustness = pd.DataFrame(cost_results)

print("\n=== Transaction Cost Robustness ===")

display(
    cost_robustness
    .pivot(
        index="Strategy",
        columns="Transaction_Cost",
        values="CAGR"
    )
    .round(4)
)

cost_robustness.to_csv(
    output_dir / "transaction_cost_robustness.csv",
    index=False
)

Running cost scenario: 0bps
Running cost scenario: 10bps
Running cost scenario: 25bps
Running cost scenario: 50bps

=== Transaction Cost Robustness ===


Transaction_Cost,0bps,10bps,25bps,50bps
Strategy,,,,
EqualWeight,0.0916,0.0915,0.0912,0.0908
GMV,0.0847,0.0840,0.0830,0.0813
MaxSharpe,0.0953,0.0921,0.0874,0.0796
RiskParity,0.0906,0.0901,0.0893,0.0881


In [4]:
# ==========================================
# Estimation-window robustness
# Common out-of-sample start date
# ==========================================

window_lengths = [
    126,   # ~6 months
    252,   # ~1 year
    504    # ~2 years
]

MAX_WINDOW = max(window_lengths)

window_results = []

# All specifications will begin their
# out-of-sample evaluation at the same date.
common_oos_start = returns.index[MAX_WINDOW]

print(
    "Common OOS start date:",
    common_oos_start
)

for window in window_lengths:

    print(
        f"\nRunning estimation window: {window}"
    )

    # Slice the data so that:
    #
    # first available OOS observation
    # = original returns.iloc[MAX_WINDOW]
    #
    # for every estimation-window length.
    aligned_returns = returns.iloc[
        MAX_WINDOW - window:
    ]

    for strategy in strategies:

        bt, _ = rolling_backtest(
            aligned_returns,
            method=strategy,
            estimation_window=window,
            holding_period=21,
            transaction_cost=0.001
        )

        m = portfolio_metrics(bt)

        window_results.append({
            "Estimation_Window": window,
            "Strategy": strategy,
            "CAGR": m["CAGR"],
            "Annualized_Volatility":
                m["Annualized_Volatility"],
            "Sharpe_rf0":
                m["Sharpe_rf0"],
            "Maximum_Drawdown":
                m["Maximum_Drawdown"]
        })


window_robustness = pd.DataFrame(
    window_results
)

print(
    "\n=== Sharpe Ratio by Estimation Window ==="
)

sharpe_window_table = (
    window_robustness
    .pivot(
        index="Strategy",
        columns="Estimation_Window",
        values="Sharpe_rf0"
    )
)

display(
    sharpe_window_table.round(4)
)

window_robustness.to_csv(
    output_dir
    / "estimation_window_robustness.csv",
    index=False
)

print(
    "\nAll estimation-window tests "
    "use the same OOS start date:",
    common_oos_start
)

Common OOS start date: 2012-01-04 00:00:00

Running estimation window: 126

Running estimation window: 252

Running estimation window: 504

=== Sharpe Ratio by Estimation Window ===


Estimation_Window,126,252,504
Strategy,,,
EqualWeight,0.7797,0.7797,0.7797
GMV,0.8960,0.8920,0.9467
MaxSharpe,0.7627,0.7223,0.8734
RiskParity,0.7549,0.8581,0.8586



All estimation-window tests use the same OOS start date: 2012-01-04 00:00:00
